In [1]:
import requests

response = requests.get(
    "https://api.frankfurter.dev/v1/latest",
    params={"base": "USD", "symbols": "EUR,GBP,PKR"}
)
print(response.status_code)
print(response.json())

200
{'amount': 1.0, 'base': 'USD', 'date': '2026-07-08', 'rates': {'EUR': 0.87689, 'GBP': 0.74917}}


## Extraction
Pulling daily exchange rates for USD → EUR, GBP, PKR from the Frankfurter API.
No transformation here — raw responses only, saved as-is for auditability.

In [2]:
def fetch_rates(target_date="latest", base_currency="USD", symbols=("EUR", "GBP", "PKR")):
    url = f"https://api.frankfurter.dev/v1/{target_date}"
    params = {"base": base_currency, "symbols": ",".join(symbols)}
    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()  # raises an error if the request failed
    return response.json()

data = fetch_rates()
data

{'amount': 1.0,
 'base': 'USD',
 'date': '2026-07-08',
 'rates': {'EUR': 0.87689, 'GBP': 0.74917}}

In [3]:
import json
from pathlib import Path
from datetime import datetime, timezone

RAW_DATA_DIR = Path("data/raw")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

def save_raw_response(payload, target_date="latest"):
    rate_date = payload.get("date", target_date)
    collected_at = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    filename = f"{rate_date}_{collected_at}.json"
    filepath = RAW_DATA_DIR / filename

    record = {
        "collected_at_utc": collected_at,
        "requested_date": target_date,
        "source": "frankfurter.dev",
        "payload": payload,
    }

    with open(filepath, "w") as f:
        json.dump(record, f, indent=2)

    print(f"Saved to {filepath}")
    return filepath

save_raw_response(data)

Saved to data\raw\2026-07-08_20260708T142356Z.json


WindowsPath('data/raw/2026-07-08_20260708T142356Z.json')

In [4]:
save_raw_response(fetch_rates())

Saved to data\raw\2026-07-08_20260708T142357Z.json


WindowsPath('data/raw/2026-07-08_20260708T142357Z.json')

In [5]:
def validate_payload(payload):
    assert "rates" in payload, "Missing 'rates' key"
    assert "date" in payload, "Missing 'date' key"
    assert len(payload["rates"]) > 0, "Empty rates dict"
    print("Payload looks valid ✓")

validate_payload(data)

Payload looks valid ✓


In [6]:
import time
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

MAX_RETRIES = 3
BACKOFF_SECONDS = 2

class ExtractionError(Exception):
    """Raised when the API can't be reached after all retries."""
    pass

def fetch_rates_with_retry(target_date="latest", base_currency="USD", symbols=("EUR", "GBP", "PKR")):
    url = f"https://api.frankfurter.dev/v1/{target_date}"
    params = {"base": base_currency, "symbols": ",".join(symbols)}

    last_exception = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            logger.info(f"Fetching rates (attempt {attempt}/{MAX_RETRIES})")
            response = requests.get(url, params=params, timeout=10)
            response.raise_for_status()
            payload = response.json()

            if "rates" not in payload:
                raise ExtractionError(f"Unexpected response shape: {payload}")

            return payload

        except (requests.RequestException, ExtractionError) as exc:
            last_exception = exc
            logger.warning(f"Attempt {attempt} failed: {exc}")
            if attempt < MAX_RETRIES:
                sleep_time = BACKOFF_SECONDS * attempt
                logger.info(f"Retrying in {sleep_time}s...")
                time.sleep(sleep_time)

    raise ExtractionError(f"Failed after {MAX_RETRIES} attempts") from last_exception

In [7]:
data = fetch_rates_with_retry()
data

2026-07-08 07:23:57,460 [INFO] Fetching rates (attempt 1/3)


{'amount': 1.0,
 'base': 'USD',
 'date': '2026-07-08',
 'rates': {'EUR': 0.87689, 'GBP': 0.74917}}

In [8]:
# Deliberately call a bad endpoint to trigger retries and confirm backoff behavior
try:
    fetch_rates_with_retry(target_date="not-a-real-date")
except ExtractionError as e:
    print(f"\nCaught expected failure: {e}")

2026-07-08 07:23:58,260 [INFO] Fetching rates (attempt 1/3)
2026-07-08 07:23:59,139 [WARNING] Attempt 1 failed: 404 Client Error: Not Found for url: https://api.frankfurter.dev/v1/not-a-real-date?base=USD&symbols=EUR%2CGBP%2CPKR
2026-07-08 07:23:59,139 [INFO] Retrying in 2s...
2026-07-08 07:24:01,141 [INFO] Fetching rates (attempt 2/3)
2026-07-08 07:24:02,001 [WARNING] Attempt 2 failed: 404 Client Error: Not Found for url: https://api.frankfurter.dev/v1/not-a-real-date?base=USD&symbols=EUR%2CGBP%2CPKR
2026-07-08 07:24:02,001 [INFO] Retrying in 4s...
2026-07-08 07:24:06,002 [INFO] Fetching rates (attempt 3/3)
2026-07-08 07:24:06,785 [WARNING] Attempt 3 failed: 404 Client Error: Not Found for url: https://api.frankfurter.dev/v1/not-a-real-date?base=USD&symbols=EUR%2CGBP%2CPKR



Caught expected failure: Failed after 3 attempts


In [9]:
import json
from pathlib import Path

RAW_DATA_DIR = Path("data/raw")

def load_raw_files():
    records = []
    for filepath in sorted(RAW_DATA_DIR.glob("*.json")):
        with open(filepath) as f:
            records.append(json.load(f))
    return records

raw_records = load_raw_files()
print(f"Loaded {len(raw_records)} raw files")
raw_records[0]

Loaded 4 raw files


{'collected_at_utc': '20260707T215638Z',
 'requested_date': 'latest',
 'source': 'frankfurter.dev',
 'payload': {'amount': 1.0,
  'base': 'USD',
  'date': '2026-07-07',
  'rates': {'EUR': 0.87466, 'GBP': 0.74706}}}

In [10]:
def flatten_record(record):
    """
    Turn one raw record into a list of clean rows — one row per currency pair.
    e.g. one API call covering EUR, GBP, PKR becomes 3 rows.
    """
    payload = record["payload"]
    rate_date = payload["date"]
    base = payload["base"]
    collected_at = record["collected_at_utc"]

    rows = []
    for currency, rate in payload["rates"].items():
        rows.append({
            "date": rate_date,
            "base_currency": base,
            "target_currency": currency,
            "rate": rate,
            "collected_at_utc": collected_at,
        })
    return rows

# test on the first record only
test_rows = flatten_record(raw_records[0])
test_rows

[{'date': '2026-07-07',
  'base_currency': 'USD',
  'target_currency': 'EUR',
  'rate': 0.87466,
  'collected_at_utc': '20260707T215638Z'},
 {'date': '2026-07-07',
  'base_currency': 'USD',
  'target_currency': 'GBP',
  'rate': 0.74706,
  'collected_at_utc': '20260707T215638Z'}]

In [11]:
import pandas as pd

EXPECTED_CURRENCIES = {"EUR", "GBP", "PKR"}

def flatten_all_records(records):
    all_rows = []
    missing_log = []

    for record in records:
        rows = flatten_record(record)
        present_currencies = {row["target_currency"] for row in rows}
        missing = EXPECTED_CURRENCIES - present_currencies

        if missing:
            missing_log.append({
                "date": record["payload"]["date"],
                "missing_currencies": sorted(missing),
            })

        all_rows.extend(rows)

    return pd.DataFrame(all_rows), missing_log

df, missing_log = flatten_all_records(raw_records)
df

2026-07-08 07:24:07,188 [INFO] NumExpr defaulting to 8 threads.


,date,base_currency,target_currency,rate,collected_at_utc
0,2026-07-07,USD,EUR,0.87466,20260707T215638Z
1,2026-07-07,USD,GBP,0.74706,20260707T215638Z
2,2026-07-07,USD,EUR,0.87466,20260707T215744Z
3,2026-07-07,USD,GBP,0.74706,20260707T215744Z
4,2026-07-08,USD,EUR,0.87689,20260708T142356Z
5,2026-07-08,USD,GBP,0.74917,20260708T142356Z
6,2026-07-08,USD,EUR,0.87689,20260708T142357Z
7,2026-07-08,USD,GBP,0.74917,20260708T142357Z


In [12]:
missing_log

[{'date': '2026-07-07', 'missing_currencies': ['PKR']},
 {'date': '2026-07-07', 'missing_currencies': ['PKR']},
 {'date': '2026-07-08', 'missing_currencies': ['PKR']},
 {'date': '2026-07-08', 'missing_currencies': ['PKR']}]

### Handling missing currencies

Frankfurter's free tier occasionally omits PKR from multi-currency responses
(observed on 2026-07-07 across multiple collections). Rather than forward-filling
or estimating a rate, missing currency-date pairs are simply left absent from
the dataset and logged separately in `missing_log`. This keeps the dataset
honest — every row reflects an actual API-confirmed rate, not an assumption.

In [14]:
def add_pct_change(df):
    df = df.sort_values(["target_currency", "date"]).copy()
    df["pct_change"] = df.groupby("target_currency")["rate"].pct_change() * 100
    return df

df = add_pct_change(df)
df

,date,base_currency,target_currency,rate,collected_at_utc,pct_change
0,2026-07-07,USD,EUR,0.87466,20260707T215744Z,NaN
1,2026-07-08,USD,EUR,0.87689,20260708T142357Z,0.254956
2,2026-07-07,USD,GBP,0.74706,20260707T215744Z,NaN
3,2026-07-08,USD,GBP,0.74917,20260708T142357Z,0.282441


In [15]:
def dedupe_by_date(df):
    """
    Keep only the latest collection per (date, currency) pair, so
    pct_change reflects true day-over-day movement, not repeated
    same-day collections during testing/backfills.
    """
    df = df.sort_values("collected_at_utc")
    df = df.drop_duplicates(subset=["date", "target_currency"], keep="last")
    return df.sort_values(["target_currency", "date"]).reset_index(drop=True)

df = dedupe_by_date(df)
df = add_pct_change(df)
df

,date,base_currency,target_currency,rate,collected_at_utc,pct_change
0,2026-07-07,USD,EUR,0.87466,20260707T215744Z,NaN
1,2026-07-08,USD,EUR,0.87689,20260708T142357Z,0.254956
2,2026-07-07,USD,GBP,0.74706,20260707T215744Z,NaN
3,2026-07-08,USD,GBP,0.74917,20260708T142357Z,0.282441


In [16]:
import sqlite3

DB_PATH = Path("data/currency_rates.db")

def load_to_sqlite(df, db_path=DB_PATH):
    conn = sqlite3.connect(db_path)

    conn.execute("""
        CREATE TABLE IF NOT EXISTS exchange_rates (
            date TEXT NOT NULL,
            base_currency TEXT NOT NULL,
            target_currency TEXT NOT NULL,
            rate REAL NOT NULL,
            pct_change REAL,
            collected_at_utc TEXT NOT NULL,
            PRIMARY KEY (date, base_currency, target_currency)
        )
    """)

    # upsert: avoids duplicate rows if this runs more than once for the same date
    df.to_sql("exchange_rates_staging", conn, if_exists="replace", index=False)
    conn.execute("""
        INSERT OR REPLACE INTO exchange_rates
        SELECT date, base_currency, target_currency, rate, pct_change, collected_at_utc
        FROM exchange_rates_staging
    """)
    conn.execute("DROP TABLE exchange_rates_staging")
    conn.commit()
    conn.close()
    print(f"Loaded {len(df)} rows into {db_path}")

load_to_sqlite(df)

Loaded 4 rows into data\currency_rates.db


In [17]:
load_to_sqlite(df)

Loaded 4 rows into data\currency_rates.db


In [18]:
conn = sqlite3.connect(DB_PATH)
result = pd.read_sql("SELECT * FROM exchange_rates ORDER BY date, target_currency", conn)
conn.close()
result

,date,base_currency,target_currency,rate,pct_change,collected_at_utc
0,2026-07-07,USD,EUR,0.87466,NaN,20260707T215744Z
1,2026-07-07,USD,GBP,0.74706,NaN,20260707T215744Z
2,2026-07-08,USD,EUR,0.87689,0.254956,20260708T142357Z
3,2026-07-08,USD,GBP,0.74917,0.282441,20260708T142357Z


In [19]:
# check what's actually in the dataframe before it hit SQLite
df[["target_currency", "rate", "pct_change"]]

,target_currency,rate,pct_change
0,EUR,0.87466,NaN
1,EUR,0.87689,0.254956
2,GBP,0.74706,NaN
3,GBP,0.74917,0.282441


In [20]:
def run_pipeline():
    """
    Full ETL run: extract latest rates, save raw, transform all
    historical raw files, and load into SQLite.
    """
    # Extract
    payload = fetch_rates_with_retry()
    save_raw_response(payload, target_date="latest")

    # Transform (reprocesses all raw files, so the table always
    # reflects the complete history, not just today's new file)
    raw_records = load_raw_files()
    df, missing_log = flatten_all_records(raw_records)
    df = dedupe_by_date(df)
    df = add_pct_change(df)

    if missing_log:
        print(f"⚠ Missing currencies detected: {missing_log}")

    # Load
    load_to_sqlite(df)

    return df

result_df = run_pipeline()
result_df

2026-07-08 07:24:45,552 [INFO] Fetching rates (attempt 1/3)


Saved to data\raw\2026-07-08_20260708T142446Z.json
⚠ Missing currencies detected: [{'date': '2026-07-07', 'missing_currencies': ['PKR']}, {'date': '2026-07-07', 'missing_currencies': ['PKR']}, {'date': '2026-07-08', 'missing_currencies': ['PKR']}, {'date': '2026-07-08', 'missing_currencies': ['PKR']}, {'date': '2026-07-08', 'missing_currencies': ['PKR']}]
Loaded 4 rows into data\currency_rates.db


,date,base_currency,target_currency,rate,collected_at_utc,pct_change
0,2026-07-07,USD,EUR,0.87466,20260707T215744Z,NaN
1,2026-07-08,USD,EUR,0.87689,20260708T142446Z,0.254956
2,2026-07-07,USD,GBP,0.74706,20260707T215744Z,NaN
3,2026-07-08,USD,GBP,0.74917,20260708T142446Z,0.282441


In [1]:
from src.extract import fetch_rates_with_retry, save_raw_response, load_raw_files
from src.transform_load import flatten_all_records, dedupe_by_date, add_pct_change, load_to_sqlite

def run_pipeline():
    payload = fetch_rates_with_retry()
    save_raw_response(payload, target_date="latest")

    raw_records = load_raw_files()
    df, missing_log = flatten_all_records(raw_records)
    df = dedupe_by_date(df)
    df = add_pct_change(df)

    if missing_log:
        print(f"⚠ Missing currencies detected: {missing_log}")

    load_to_sqlite(df)
    return df

result_df = run_pipeline()
result_df

2026-07-08 08:26:16,142 [INFO] NumExpr defaulting to 8 threads.
2026-07-08 08:26:16,856 [INFO] Fetching rates (attempt 1/3)
2026-07-08 08:26:18,445 [INFO] Saved raw response to C:\Users\Daniyal\etl-currency\data\raw\2026-07-08_20260708T152618Z.json


⚠ Missing currencies detected: [{'date': '2026-07-07', 'missing_currencies': ['PKR']}, {'date': '2026-07-07', 'missing_currencies': ['PKR']}, {'date': '2026-07-08', 'missing_currencies': ['PKR']}, {'date': '2026-07-08', 'missing_currencies': ['PKR']}, {'date': '2026-07-08', 'missing_currencies': ['PKR']}, {'date': '2026-07-08', 'missing_currencies': ['PKR']}]
Loaded 4 rows into C:\Users\Daniyal\etl-currency\data\currency_rates.db


,date,base_currency,target_currency,rate,collected_at_utc,pct_change
0,2026-07-07,USD,EUR,0.87466,20260707T215744Z,NaN
1,2026-07-08,USD,EUR,0.87689,20260708T152618Z,0.254956
2,2026-07-07,USD,GBP,0.74706,20260707T215744Z,NaN
3,2026-07-08,USD,GBP,0.74917,20260708T152618Z,0.282441
